In [ ]:
import os
import numpy as np, pandas as pd
from scipy import stats
from scipy.stats import t as tdist

from google.colab import drive
drive.mount('/content/drive')

SRC = '/content/drive/MyDrive/BOOSTMAP/데이터/완료'
OUT = '/content/drive/MyDrive/BOOSTMAP/데이터/완료/가설검정'
os.makedirs(OUT, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── 데이터 로드 ────────────────────────────────────────
A = pd.read_csv(f'{SRC}/model_A_financial.csv')
B = pd.read_csv(f'{SRC}/model_B_nonfinancial.csv')
C = pd.read_csv(f'{SRC}/model_C_nonfinancial_new.csv')

SKIP = {'SK_ID_CURR', 'TARGET', 'AGE_BAND'}
gof = {}
for g, d in [('A_금융', A), ('B_비금융_기존', B), ('C_비금융_신규', C)]:
    for c in d.columns:
        if c not in SKIP:
            gof[c] = g

X = pd.concat([A, B, C], axis=1).loc[:, list(gof)]
y = A['TARGET'].values.astype(float)
n = len(X)
print(f'로드 완료 — 변수 {X.shape[1]}개 / 행 {n:,} / 연체율 {y.mean()*100:.2f}%')


로드 완료 — 변수 69개 / 행 307,511 / 연체율 8.07%


In [ ]:
# ── 상관 검정   H0: rho = 0  (연체와 상관이 없다) ──────
rows = []
for c in X.columns:
    v = X[c].values.astype(float)
    if np.std(v) == 0:
        continue
    rp, pp = stats.pearsonr(v, y)      # 연속형=점이연 / 이진형=phi
    rs, ps = stats.spearmanr(v, y)     # 순위 기반 (비선형 대응)
    rows.append({'변수': c, '군': gof[c],
                 '유형': '이진' if X[c].nunique() <= 2 else '연속',
                 '피어슨_r': round(rp, 4),     '피어슨_p': pp,
                 '스피어만_rho': round(rs, 4), '스피어만_p': ps})

res = pd.DataFrame(rows)
res['피어슨_유의']   = res['피어슨_p']   < 0.05
res['스피어만_유의'] = res['스피어만_p'] < 0.05
res['방향'] = np.where(res['피어슨_r'] > 0, '위험↑', '위험↓')
res['차이'] = (res['스피어만_rho'].abs() - res['피어슨_r'].abs()).round(4)

In [ ]:
# ── 결과 ───────────────────────────────────────────────
print(f'\n[1] H0: rho = 0  기각 비율')
for k, col in [('피어슨', '피어슨_유의'), ('스피어만', '스피어만_유의')]:
    print(f'  {k:5s} {res[col].sum():2d}/{len(res)}  ({res[col].mean()*100:.1f}%)')
print(f"\n  |r| 최대 {res['피어슨_r'].abs().max():.4f}"
      f"  중앙 {res['피어슨_r'].abs().median():.4f}"
      f"  | 0.1 미만 {int((res['피어슨_r'].abs() < 0.1).sum())}개")

print('\n[2] |r| 상위 15 — 연체와 관계가 큰 변수')
print(res.reindex(res['피어슨_r'].abs().sort_values(ascending=False).index)
         .head(15)[['변수', '군', '유형', '피어슨_r', '스피어만_rho', '방향']]
         .to_string(index=False))

print('\n[3] H0을 기각하지 못한 변수')
ns = res[~res['피어슨_유의']][['변수', '군', '피어슨_r', '피어슨_p']]
print(ns.to_string(index=False) if len(ns) else '  없음')

print('\n[4] 군별 |r| 평균')
print(res.groupby('군')[['피어슨_r', '스피어만_rho']]
         .apply(lambda d: d.abs().mean()).round(4).to_string())

print('\n[5] 피어슨 vs 스피어만 차이 상위 8 — 비선형 신호')
print(res.reindex(res['차이'].abs().sort_values(ascending=False).index)
         .head(8)[['변수', '군', '피어슨_r', '스피어만_rho', '차이']].to_string(index=False))

# ── 표본 크기가 유의성에 미치는 영향 ───────────────────
tcrit = tdist.ppf(0.975, n - 2)
rcrit = tcrit / np.sqrt(tcrit**2 + n - 2)
print(f'\n[6] n={n:,}에서 p<0.05가 되는 최소 |r| = {rcrit:.5f}')

res.to_csv(f'{OUT}/상관검정_TARGET.csv', index=False, encoding='utf-8-sig')
print(f'\n저장 -> {OUT}/상관검정_TARGET.csv')


[1] H0: rho = 0  기각 비율
  피어슨   60/69  (87.0%)
  스피어만  58/69  (84.1%)

  |r| 최대 0.0782  중앙 0.0128  | 0.1 미만 69개

[2] |r| 상위 15 — 연체와 관계가 큰 변수
                                                 변수        군 유형   피어슨_r  스피어만_rho  방향
                                                AGE B_비금융_기존 연속 -0.0782   -0.0783 위험↓
                                          LTV_GOODS     A_금융 연속  0.0694    0.0668 위험↑
                         NAME_INCOME_TYPE_G_Working B_비금융_기존 이진  0.0575    0.0575 위험↑
             NAME_EDUCATION_TYPE_C_Higher education C_비금융_신규 이진 -0.0566   -0.0566 위험↓
NAME_EDUCATION_TYPE_C_Secondary / secondary special C_비금융_신규 이진  0.0498    0.0498 위험↑
                                     YEARS_EMPLOYED B_비금융_기존 연속 -0.0461   -0.0252 위험↓
                                 DAYS_EMPLOYED_ANOM B_비금융_기존 이진 -0.0460   -0.0460 위험↓
                                  BUREAU_DEBT_RATIO     A_금융 연속  0.0459    0.0540 위험↑
                           BUREAU_ACTIVE_LOAN_COUNT     A_금융 연속  0.0436    0.0284 위험